### Step 1: Creating Tokens


In [20]:
with open("E:\python\LLM-From-Scratch\dataset\Book1.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()


print("Total number of characters: ", len(raw_text))
print(raw_text[:100])

Total number of characters:  474422
THE BOY WHO LIVED 

Mr. and Mrs. Dursley, of number four, Privet Drive, 
were proud to say that they


In [21]:
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:100])
print(len(preprocessed))

['THE', 'BOY', 'WHO', 'LIVED', 'Mr', '.', 'and', 'Mrs', '.', 'Dursley', ',', 'of', 'number', 'four', ',', 'Privet', 'Drive', ',', 'were', 'proud', 'to', 'say', 'that', 'they', 'were', 'perfectly', 'normal', ',', 'thank', 'you', 'very', 'much', '.', 'They', 'were', 'the', 'last', 'people', 'you’d', 'expect', 'to', 'be', 'involved', 'in', 'anything', 'strange', 'or', 'mysterious', ',', 'because', 'they', 'just', 'didn’t', 'hold', 'with', 'such', 'nonsense', '.', 'Mr', '.', 'Dursley', 'was', 'the', 'director', 'of', 'a', 'firm', 'called', 'Grunnings', ',', 'which', 'made', 'drills', '.', 'He', 'was', 'a', 'big', ',', 'beefy', 'man', 'with', 'hardly', 'any', 'neck', ',', 'although', 'he', 'did', 'have', 'a', 'very', 'large', 'mustache', '.', 'Mrs', '.', 'Dursley', 'was', 'thin']
99426


### Step 2: Creating Token IDs

In [22]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

7733


In [23]:
vocab = {token:integer for integer, token in enumerate(all_words)}

for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('-', 6)
('-J', 7)
('-bodied', 8)
('.', 9)
('/', 10)
('0', 11)
('1', 12)
('10', 13)
('100', 14)
('101', 15)
('102', 16)
('103', 17)
('104', 18)
('105', 19)
('106', 20)
('107', 21)
('108', 22)
('109', 23)
('11', 24)
('110', 25)
('111', 26)
('112', 27)
('113', 28)
('114', 29)
('115', 30)
('116', 31)
('117', 32)
('118', 33)
('119', 34)
('12', 35)
('120', 36)
('121', 37)
('122', 38)
('123', 39)
('124', 40)
('125', 41)
('126', 42)
('127', 43)
('128', 44)
('129', 45)
('13', 46)
('130', 47)
('131', 48)
('132', 49)
('133', 50)


### Step 3: Creating the Encode and Decode Block

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        
        return text

In [38]:
tokenizer = SimpleTokenizerV1(vocab)


text = """ 
        "I hope you’re pleased with yourselves. We could all 
        have been killed — or worse, expelled. Now, if you 
        don’t mind, I’m going to bed.” 
       """

ids = tokenizer.encode(text)
print(ids)

[1, 909, 3872, 7074, 4979, 6965, 7070, 9, 1542, 2583, 1700, 3762, 1947, 4102, 7083, 4736, 7006, 5, 3138, 9, 1127, 5, 3950, 7062, 2884, 4460, 5, 934, 3595, 6463, 1938, 9, 7730]


In [39]:
tokenizer.decode(ids)

'" I hope you’re pleased with yourselves. We could all have been killed — or worse, expelled. Now, if you don’t mind, I’m going to bed. ”'

### Adding Special Tokens

In [42]:
from numpy import integer


all_tokens = sorted(list(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [43]:
len(vocab.items())

7735

In [44]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('”', 99421)
('•k', 99424)
('■”', 99425)
('<|endoftext|>', 99426)
('<|unk|>', 99427)


In [54]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
                        item if item in self.str_to_int 
                        else "<|unk|>" for item in preprocessed ]


        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [62]:
from idna import encode


tokenizer = SimpleTokenizerV2(vocab)

text1 = " “After all this time?”$ " 
text2 = " “Always,” said Snape. " 

text = " <|endoftext|>".join((text1, text2))

print(text)

 “After all this time?”$  <|endoftext|> “Always,” said Snape. 


In [63]:
sample = tokenizer.encode(text)
print(sample)
tokenizer.decode(sample)

[94670, 30739, 82023, 82641, 14724, 99427, 99426, 94709, 6178, 99421, 70274, 25979, 13375]


'“After all this time? <|unk|> <|endoftext|> “Always, ” said Snape.'